In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import textwrap
import google.generativeai as genai

In [ ]:
dataset = load_dataset("lavita/MedQuAD")
dataset

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})


In [ ]:
df = dataset["train"].to_pandas()

In [ ]:
df.columns

Index(['document_id', 'document_source', 'document_url', 'category',
       'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms',
       'question_id', 'question_focus', 'question_type', 'question', 'answer'],
      dtype='object')

In [ ]:
df[["question", "answer", "document_source"]].isna().sum()

,0
question,0
answer,31034
document_source,0


In [ ]:
df = df[["question", "answer", "document_source"]].dropna()
df.head()

,question,answer,document_source
0,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...,GHR
1,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...,GHR
2,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen...",GHR
3,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...,GHR
4,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...,GHR


In [ ]:
len(df)

16407

In [ ]:
documents = []

for idx, row in df.iterrows():
    text = f"""
Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append({
        "id": idx,
        "text": text,
        "source": row["document_source"],
        "question": row["question"]
    })

len(documents)

16407

In [ ]:
print(documents[0]["text"][:1000])


Question: What is (are) keratoderma with woolly hair ?

Answer: Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnorm

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
small_docs = documents

texts = [doc["text"] for doc in small_docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings.shape

Batches:   0%|          | 0/513 [00:00<?, ?it/s]

(16407, 384)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 16407


In [ ]:
def retrieve_docs(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        doc = small_docs[idx]
        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "distance": float(distance)
        })

    return results

In [ ]:
query = "What are the symptoms of diabetes?"

results = retrieve_docs(query, top_k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc["source"])
    print("Matched Question:", doc["question"])
    print("Distance:", doc["distance"])
    print(doc["text"][:700])


--- Result 1 ---
Source: NIHSeniorHealth
Matched Question: What are the symptoms of Diabetes ?
Distance: 0.42415088415145874

Question: What are the symptoms of Diabetes ?

Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, loss of feeling or tingling in the feet and blurry eyesight. Some people with diabetes, however, have no symptoms at all.


--- Result 2 ---
Source: NIHSeniorHealth
Matched Question: What are the symptoms of Diabetes ?
Distance: 0.4445371627807617

Question: What are the symptoms of Diabetes ?

Answer: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 

In [ ]:
!pip install -q langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

os.environ["GROQ_API_KEY"] = "Enter your Groq API key here"


In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:

def generate_answer(query, retrieved_docs):
    context = "\n\n".join([
        f"Source {i+1}:\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the context provided below.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "The provided medical documents do not contain enough information."
3. Keep the answer simple and clear.
4. Do not diagnose the user.
5. Suggest consulting a healthcare professional when appropriate.

Context:
{context}

User question:
{query}

Answer:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def rag_chatbot(query, top_k=3):
    retrieved_docs = retrieve_docs(query, top_k=top_k)
    answer = generate_answer(query, retrieved_docs)

    print("Question:")
    print(query)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Distance:", doc["distance"])

In [ ]:
rag_chatbot("How is high blood pressure treated?")

Question:
How is high blood pressure treated?

Answer:
High blood pressure is treated with lifestyle changes and medicines. Treatment can help control blood pressure, but it will not cure high blood pressure, even if your blood pressure readings appear normal. 

Lifestyle changes include making ongoing medical care a priority. 

Medicines work in different ways, such as removing extra fluid and salt from your body, slowing down the heartbeat, or relaxing and widening blood vessels. Some common types of medicines used to treat high blood pressure include:

- Diuretics (water or fluid Pills)
- Beta Blockers
- Angiotensin-Converting Enzyme (ACE) Inhibitors
- Angiotensin II Receptor Blockers (ARBs)
- Calcium Channel Blockers
- Alpha Blockers
- Alpha-Beta Blockers
- Central Acting Agents
- Vasodilators

It's recommended to work with your healthcare team for lifelong blood pressure control.

Retrieved Sources:

--- Source 1 ---
Dataset Source: NIHSeniorHealth
Matched Question: What are the t